## Semantic Text Search : base approach

Text Search is important concept for many applications, the idea to develop text search programatically may differ based on the Application, i.e google search far different compared to searching products in e-commerce application.

The key-word based search, which returns a part of the text that best matches with the searched query, is still used in small scale applications, as the application does not have much data they could get better results with minor spell correction techniques.
But for an application that contains large content of text, the key-word match with spell correction is not always perform well, 

let's understand step by step :

Imagine we have a e-commerce fashion wear website, not small & not large, and we need a Text search algorithm for users to search for products based on their interest.

The first approach is,

The user will search with the some text, like `black cotton shirt`, then we convert this sentence into words, and we will search into all product's description & title in which all the three words can be found..

this could work well only if all the spellings of the words are correct.

So, this is just a basic approach, it has to handle multiple cases.. let's see few of them.

* Case 1:
If the spelling of any word is wrong, either the word can be found in the product information or it may not. Even though we find the product information that matching the search string, it may not be as expected or meaningful as few words of the search string are misspeled.

* Case 11:
If the misspelled words are not an english word, or not a domain specific word(user may search with brand name which must not be and english and must have meaning), then we have to form or guess a new word to that misspelled word, there are code libraries to correct spelling for a misspelled word.

>> let's see an example :

The search string "blck shirt" has a misspelled word "blck" which may not match any product information, even though it is helpful to user if we provide possible matches, because the user may mistyped the words, or the user don't know the correct spelling, so we have to correct the spelling and search the possible words..

In [6]:
from spellchecker import SpellChecker

spell = SpellChecker() # distance=1
word_list = "blck shirt".split(" ")
misspelled = spell.unknown(word_list)

for word in misspelled:
    print(spell.candidates(word))
    print(spell.correction(word))

{'black', 'buck', 'block', 'blak', 'bock', 'back', 'beck'}
back


In the above code, we have applied spell correction to the "blck", it has multiple suggestions for word "blck", but it has given "back" as final solution, which is not as expected.
So, it is very important for our program to keep the application specific knowledge to produce better results.

Let's see how..

<!-- The simple way to handle this is, collect all the unique words from your product information, and calculate leveshtien distance to all the words, and pick one. -->

In [19]:
import spacy

l = ['black', 'buck', 'block', 'blak', 'bock', 'back', 'beck']

nlp = spacy.load('en_core_web_md')

token2 = nlp("shirt")

for i in d.suggest(word):
    print(i, token2, " ~~ Similarity:", nlp(i).similarity(token2))

blk shirt  ~~ Similarity: 0.06794350134286951
black shirt  ~~ Similarity: 0.48911827929682505
block shirt  ~~ Similarity: 0.14079983132434976
beck shirt  ~~ Similarity: 0.20174259521134344
back shirt  ~~ Similarity: 0.3764558244187314
bock shirt  ~~ Similarity: 0.15182986560280537
buck shirt  ~~ Similarity: 0.27525302704318216
bl ck shirt  ~~ Similarity: 0.19410031852434875
bl-ck shirt  ~~ Similarity: 0.21242972675826022


So, here in the above code I have used spacy's NLP library to find word similarity between two strings, here I will calculate similarity between the possible spell correction words of a misspelled word and the remaining string.

So, here the similarity between "black" and "shirt" is high.

Now, there is another issue, there may not be any spelling mistakes but the words are mistyped unintentionally.. like "black" typed as "blak", to solve that we use same technique that we used above, but we use enchant library here as it will give all phonetic(pronounciation) based words.

In [20]:
import enchant
import spacy
  
nlp = spacy.load('en_core_web_md')

d = enchant.Dict("en_US")
word = "blak"

token2 = nlp("shirt")

for i in d.suggest(word):
    print(i, token2, " ~~ Similarity:", nlp(i).similarity(token2))

balk shirt  ~~ Similarity: 0.12950914684729442
blk shirt  ~~ Similarity: 0.06794350134286951
bleak shirt  ~~ Similarity: 0.08433008209095842
blank shirt  ~~ Similarity: 0.3208267925770709
black shirt  ~~ Similarity: 0.48911827929682505
beak shirt  ~~ Similarity: 0.2016826586462395
blat shirt  ~~ Similarity: 0.12914789458755888
blag shirt  ~~ Similarity: 0.08914622488592719
blah shirt  ~~ Similarity: 0.20806968316823962
blab shirt  ~~ Similarity: -0.012738009815627102
flak shirt  ~~ Similarity: 0.17784370599305258
Blake shirt  ~~ Similarity: 0.2066370684147796


So, we can get the better black shirt as the most similar pair.

* There could be another case where almost all of the words are misspelled, in that case we should calculate similarity in the starting, first we should find best possible english words for them(using domain specific  words is recommended), then we re check the importance of all words by finding similarity and replace then if there are any better words..

In [48]:
import time
import os
import json
import fuzzy
import enchant
import Levenshtein
import numpy as np
from spellchecker import SpellChecker
import itertools

def choose_better_word(misword, rem_doc, old_sim, chanted_words_list):
    # d = enchant.Dict("en_US")
    # chanted_words_list = d.suggest(misword)  # chanted_words(misword)
    # print(chanted_words_list)
    sim_list = []
    for w1 in chanted_words_list:
        w1 = nlp(w1)
        w2 = rem_doc #nlp(rem_doc)
        sim = w1.similarity(w2)
        # print(w1, w2, sim)
        sim_list.append(sim)
    # print(sim_list)
    max_sim = max(sim_list)
    if max_sim > old_sim:
        sim_list_max_idx = np.argmax(sim_list)
        new_word = chanted_words_list[sim_list_max_idx]
    else:
        new_word = misword
    # print('new_word : ', new_word)
    return new_word

def best_match(my_words, word, distance=2):
    dists = []
    close_words = []
    sound_words = []
    suggested_words = []
    soundex = fuzzy.Soundex(4)
    for i in my_words:
        dist = Levenshtein.distance(i.lower(), word)
        if dist <= distance and soundex(i)[0] == soundex(word)[0]:
            dists.append(dist)
            close_words.append(i)
        if soundex(i) == soundex(word):
            if i not in close_words:
                sound_words.append(i)
    
    if len(dists) > 0:
        inds = np.argsort(dists)
        suggested_words = [close_words[i] for i in inds]
    print('leven ---------->> ', suggested_words, ' <<------------')
    if len(suggested_words) == 0:
        suggested_words = sound_words
    print('sound ---------->> ', sound_words, ' <<------------')
    return suggested_words

# today test (APril 14th ~ KGF_2 release date..)

def testing_search(request):
#     {
#         "query" : "lodge balence"
#     }
    # ict riconsilation
    # bilens shit repot
    # lodge balence
    # finence repot
    begin = time.time()
    
    searched_str = request #request.data['query']
    word_list = searched_str.split(" ")

#     module_dir = os.path.dirname(__file__)
#     file_path = os.path.join(module_dir, 'my_words.json')   #full path to text
    file_path = "my_words.json"
    with open(file_path, 'r') as file:
        my_words = json.loads(file.read())['my_words']

    spell = SpellChecker() # distance=1
    misspelled = spell.unknown(word_list)

    final_words_list = []

    for word in word_list:
        if word not in misspelled:
            final_words_list.append(word)
        else:
            bm = best_match(my_words, word, distance=2)
            if len(bm) > 0:
                # print(bm)
                final_words_list.append(bm[0])
            else:
                # print(spell.candidates(word))
                # final_words_list.append(spell.correction(word))
                final_words_list.append(word)

    ex_str = ' '.join(final_words_list)

    searched_text = ex_str #"find nearby paks"

    words_list = searched_text.split(" ")

    all_words = []

    for i in words_list:
        temp = [i]
        temp += best_match(my_words, i, 2)
        temp = list(set(temp))
        all_words.append(temp)
    
    print(all_words, '<<<<<<<<<<<<<<')

    all_combos = list(itertools.product(*all_words))

    print(len(all_combos), all_combos)

    scores = []
    for combo in all_combos:
        this_score = []
        for w1 in combo:
            w2 = ' '.join([item for item in combo if item != w1])
            this_score.append(nlp(w1).similarity(nlp(w2)))
        scores.append(max(this_score))
    
    print(scores, np.argsort(scores))
    
    max_sim_score = scores[np.argmax(scores)]
    
    best_score_inds = []
    for val in range(len(scores)):
        if scores[val] == max_sim_score:
            best_score_inds.append(val)

    # final_words = []
    final_words = list(all_combos[best_score_inds[-1]])
    
    final_str = ' '.join(final_words)
    
    end = time.time()
    run_time = str((end - begin)) + ' sec'
    return {'run_time': run_time, 'final_str': final_str}

In [51]:
testing_search("lether wach")

leven ---------->>  ['leather']  <<------------
sound ---------->>  []  <<------------
leven ---------->>  ['watch']  <<------------
sound ---------->>  []  <<------------
[['lether', 'leather'], ['wach', 'watch']] <<<<<<<<<<<<<<
4 [('lether', 'wach'), ('lether', 'watch'), ('leather', 'wach'), ('leather', 'watch')]
[0.023655023714787997, 0.24787827368997487, 0.023655023714787997, 0.24787827368997487] [0 2 1 3]


{'run_time': '0.2679440975189209 sec', 'final_str': 'leather watch'}